In [0]:
%run ./transform_data

Import table DIM_BATCH

Import table FACT_RECOMMENDATIONS

cette table doit contenir les informations suivantes : 
- missing_values
- batch_id
- id_recommendation
- alerte_id
- flg_succes
- flg_echec
- recos_attendue
- type_missing_values
- flg_auto
- flg_manual
- date saisie manuelle

exploser les valeurs missing values en une seule colonne : missing_values (regrouper valeurs auto et manuelles au même endroit)

In [0]:
exploded_auto_values = max_date.withColumn(
    "exploded_array",
    F.when(
        F.size(F.col("automatic_missing_values")) == 0, 
        F.array(F.lit("NO_VALUE"))
    ).when(
        F.col("automatic_missing_values").isNull(),
        F.array(F.lit("NO_VALUE"))
    ).otherwise(F.col("automatic_missing_values"))
).withColumn(
    "missing_value",
    F.explode_outer(F.col("exploded_array"))  # Exploser le tableau pour créer une ligne par valeur
).withColumn(
    "flg_auto",
    F.when(
        (F.col("automatic_missing_values").isNull()) | 
        (F.size(F.col("automatic_missing_values")) == 0), 
        0
    ).otherwise(1)  # Si le tableau contient des éléments, flg_auto = 1
).withColumn(
    "flg_manual", # utilisé pour la jointure d'après 
    F.lit(0)  # Initialisation de flg_manual à 0
)
# Sélectionner les colonnes finales
exploded_auto = exploded_auto_values.select(
    "batch_id",
    "production_line",
    "calculation_interval_after",
    "job_execution_datetetime",
    "target_localization",
    "is_success",
    "missing_value",
    "flg_auto",
    "flg_manual",
    "batch_status",
    "overlap"
)

In [0]:
exploded_manual_values = max_date.withColumn(
    "exploded_array",
    F.when(
        F.size(F.col("manual_missing_values")) == 0, 
        F.array(F.lit("NO_VALUE"))
    ).when(
        F.col("manual_missing_values").isNull(),
        F.array(F.lit("NO_VALUE"))
    ).otherwise(F.col("manual_missing_values"))
).withColumn(
    "missing_value",
    F.explode_outer(F.col("exploded_array"))  # Exploser le tableau pour créer une ligne par valeur
).withColumn(
    "flg_manual",
    F.when(
        (F.col("manual_missing_values").isNull()) | 
        (F.size(F.col("manual_missing_values")) == 0), 
        0
    ).otherwise(1)  # Si le tableau contient des éléments, flg_auto = 1
).withColumn(
    "flg_auto", # utilisé pour la jointure d'après 
    F.lit(0)  # Initialisation de flg_manual à 0
)

# Sélectionner les colonnes finales
exploded_manual = exploded_manual_values.select(
    "batch_id",
    "production_line",
    "calculation_interval_after",
    "job_execution_datetetime",
    "target_localization",
    "is_success",
    "missing_value",
    "flg_auto",
    "flg_manual",
    "batch_status",
    "overlap"
)

In [0]:
union_exploded_values_bis = (
    exploded_auto
    .unionByName(exploded_manual) )

union_exploded_values = union_exploded_values_bis.withColumn(
    "type_de_missing_value",
    F.when(F.col("flg_auto") == 1, "automatique")
    .when(F.col("flg_manual") == 1, "manuelle")
    .otherwise(None)
)

union_exploded_values =  union_exploded_values.dropDuplicates()


In [0]:

# Créer une colonne pour marquer les lignes où missing_value est différent de "NO_VALUE"
union_exploded_values = union_exploded_values.withColumn(
    "has_valid_value",
    when(col("missing_value") != "NO_VALUE", 1).otherwise(0)
)

# Définir une fenêtre partitionnée par batch_id et target_localization
window_spec = Window.partitionBy("batch_id", "target_localization", "production_line")

# Ajouter une colonne pour indiquer s'il existe au moins une ligne avec missing_value différent de "NO_VALUE"
union_exploded_values = union_exploded_values.withColumn(
    "has_valid_value_in_group",
    max("has_valid_value").over(window_spec)
)

# Filtrer les lignes où missing_value est "NO_VALUE" et il existe au moins une autre ligne avec missing_value différent de "NO_VALUE"
union_exploded_values_filtered = union_exploded_values.filter(
    ~((col("missing_value") == "NO_VALUE") & (col("has_valid_value_in_group") == 1))
)

# Supprimer les colonnes temporaires
union_exploded_values_filtered = union_exploded_values_filtered.drop("has_valid_value", "has_valid_value_in_group")


Maintenant, on peut faire la jointure entre les données inference monitoring (union_monitoring_with_translate_tg) et les données des recommandations (join_reco_batch_planning)

In [0]:
# on récupère l'id_parameter_localization de activities pour anticiper la jointure entre les données d'inference monitoring et des recommandations pg

union_monitoring_with_id_localization = union_exploded_values_filtered.alias("a").join(
    activity_mapping.alias("b"),
    F.col("a.target_localization") == F.col("b.code"),
    "left").select(
        F.col("a.*"),
        F.col("b.id_parameter_localization")
    )

In [0]:
# table principale = liste des batchs de batch_planning avec les tg attendues et les données d'inférence en table secondaire

join_candidates_with_recommandation = succes_reco_count_distinct.alias("a").join( 
    union_monitoring_with_id_localization.alias("b"), 
    (F.col("a.batch_id") == F.col("b.batch_id")) &
    (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")),
    "left").select(
        "a.id_production_line",
        "b.production_line",
        "a.batch_id",
        "a.id_recommendation",
        "a.id_parameter_localization",
        "a.date_creation_reco",
        "a.obsolescence_status",
        "a.flg_lasted_reco",
        "b.batch_status",
     F.when(F.col("b.calculation_interval_after").isNotNull(), F.col("b.calculation_interval_after"))
     .otherwise(F.col("a.calculation_interval_after_batch_planning"))
     .alias("calculation_interval_after_datetime"), #si valeur absente inference monitoring alors prendre celle issue de pg batch planning

        "b.missing_value",
        "b.flg_manual",
        "b.flg_auto",
        "b.type_de_missing_value",
        "b.overlap",
        "job_execution_datetetime"
    )

Pour finaliser la table de faits, il faut désormais récupérer la date de saisie manuelle des valeurs manuelles. on récupère les informations depuis la table manual_entries.

In [0]:
manual_values_data = (
    manual_entries.alias("a")
    .join(
        parameters_variables.alias("b"),
        F.col("a.parameter") == F.col("b.id_parameter_variable"),
        "left"
    )
    .filter(F.col("a.deleted") == False)  # Filtrer les entrées non supprimées
    .groupBy("a.batch", "b.code")          # Regrouper par batch et code
    .agg(F.max("a.created_at").alias("date_saisie_manuelle"))  # Récupérer la max date
)

In [0]:
table_fact_entries = join_candidates_with_recommandation.alias("a").join(
    manual_values_data.alias("b"),
    (F.col("a.batch_id") == F.col("b.batch")) &
    (F.col("a.missing_value") == F.col("b.code")),
    "left"
).select(
    "a.*",
    "b.date_saisie_manuelle"
)


Maintenant que toutes les données ont été récupérées pour cette table, on va pouvoir calculer les colonnes flg_succes et flg_echec pour comptabiliser le nbr de recos en échec et en succès sans dédoublonner nos valeurs puisque la granularité la plus fine de la table est la valeur manquante. 

In [0]:
count_echec_succes = table_fact_entries.withColumn(
    "succes",
    F.when(
        F.col("id_recommendation").isNotNull() &
        (F.col("calculation_interval_after_datetime") <= F.current_timestamp()),
        1
    ).otherwise(0)
).withColumn(
    "echec",
    F.when(
        F.col("id_recommendation").isNull() &
        (F.col("calculation_interval_after_datetime") <= F.current_timestamp()),
        1
    ).otherwise(0)
)

In [0]:
ranked_values_window_spec = Window.partitionBy(
    F.col("a.batch_id"), 
    F.col("a.id_parameter_localization"),
    F.col("a.id_recommendation")
).orderBy(
    F.when(F.col("a.missing_value") != "NO_VALUE", 0).otherwise(1)
)

window_echec_succes_distinct =  count_echec_succes.withColumn(
        "rn",
        F.when(
            (F.col("missing_value") == "NO_VALUE") &
            F.col("succes").isNull() &
            F.col("echec").isNull(),
            1
        ).otherwise(
            F.row_number().over(ranked_values_window_spec)
        )
    )

succes_echec_count_distinct = window_echec_succes_distinct.withColumn(
    "flg_succes",
    F.when((F.col("id_recommendation").isNotNull()) & (F.col("rn") == 1) & (F.col("calculation_interval_after_datetime") <= F.current_timestamp()), 1).otherwise(0)
).withColumn(
    "flg_echec",
    F.when((F.col("id_recommendation").isNull()) & (F.col("rn") == 1) & (F.col("calculation_interval_after_datetime") <= F.current_timestamp()), 1).otherwise(0))


In [0]:

# filter les paires batch_id et target_localization avec plusieurs id_reco et garder le plus récent

exclude_multi_reco = succes_echec_count_distinct.filter(F.col("flg_lasted_reco") == 1)

Ajout des id_alerting pour jointure sur la table de dimension des alertes

In [0]:
batch_creation_date = exclude_multi_reco.alias("a").join(
    batches_full_list.alias("b"),
    F.col("batch_id") == F.col("b.id_batch"),
    "left").select(
    "a.*",
    "b.batch_creation_date")

In [0]:
is_missing_occurences_late_measures = (
    missing_asset_measure_with_tg_code
    .groupBy(
        "batch_id",
        "id_parameter_localization",
        "measure_name"
    )
    .agg(F.max(F.col("is_missing_occurrences")).alias("is_missing_occurrences"),
          F.max(F.col("is_late")).alias("is_late")
    )
)

In [0]:
failure_measure_is_late_tag_drop = (
    batch_creation_date.alias("a")
    .join(
        is_missing_occurences_late_measures.alias("b"),
        (F.col("a.batch_id") == F.col("b.batch_id")) &
        (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")) &
        (F.col("a.missing_value") == F.col("b.measure_name")),
        "left"
    )
    .select(
        "a.*",
        "b.is_missing_occurrences",
        "b.is_late"
    )
)

In [0]:
#agg. à la maille reco pour ajouter les alertes
is_late_for_reco = (
    failure_measure_is_late_tag_drop
    .groupBy("batch_id", "id_parameter_localization")
    .agg(
        F.max("is_late").alias("is_late_for_reco"), 
        F.max("is_missing_occurrences").alias("is_missing_occurrences_for_reco")
    )
)

In [0]:
failure_measure_is_late_tag_drop_with_agg = failure_measure_is_late_tag_drop.alias("a").join(
    is_late_for_reco.alias("b"),
        (F.col("a.batch_id") == F.col("b.batch_id")) &
        (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")),
        "left"
    ).select(
        "a.*",
        "b.is_late_for_reco",
        "b.is_missing_occurrences_for_reco"
    )

In [0]:
monitoring_candidates = failure_measure_is_late_tag_drop_with_agg.filter(
    (F.col("flg_echec") == 1) &
    (F.col("flg_auto") == 1) &
    (F.col("missing_value") != "NO_VALUE")
)

In [0]:
execution_site_pairs = (
    monitoring_candidates
    .select("job_execution_datetetime", "production_line", "batch_id")
    .filter(
        F.col("production_line").isNotNull() &
        F.col("batch_id").isNotNull() &
        F.col("job_execution_datetetime").isNotNull()
    )
)

In [0]:

execution_site_pairs_rename = execution_site_pairs.withColumn(
    "production_line",
    F.when(F.col("production_line") == "NOGENT2", "NG2")
     .when(F.col("production_line") == "NOGENT1", "NG1")
     .when(F.col("production_line") == "STRASBOURG2", "ST2")
     .when(F.col("production_line") == "STRASBOURG1", "ST1")
     .when(F.col("production_line") == "ROUEN1", "RO1")
     .when(F.col("production_line") == "PROUYY1", "PR1")
     .when(F.col("production_line") == "POLISY1", "PO1")
     .when(F.col("production_line") == "BUZAU1", "BU1")
     .when(F.col("production_line") == "BOLELEMI1", "BO1")
     .otherwise(F.col("production_line"))
)

In [0]:
site_table_mapping = {
    "NG1": f"mal_maite_nogent1_{current_environment}.gold.measurements_pivot",
    "NG2": f"mal_maite_nogent2_{current_environment}.gold.measurements_pivot",
    "PR1": f"mal_maite_prouvy1_{current_environment}.gold.measurements_pivot",
    "RO1": f"mal_maite_rouen1_{current_environment}.gold.measurements_pivot",
    "ST2": f"mal_maite_strasbourg2_{current_environment}.gold.measurements_pivot",
    "PO1": f"mal_maite_polisy1_{current_environment}.gold.measurements_pivot",
    "BU1": f"mal_maite_buzau1_{current_environment}.gold.measurements_pivot",
    "BO1": f"mal_maite_bolelemi1_{current_environment}.gold.measurements_pivot"
}

In [0]:


execution_groups = (
    monitoring_candidates
    .filter(F.col("production_line").isin(list(site_table_mapping.keys())))
    .groupBy("production_line", "job_execution_datetetime")
    .agg(
        F.collect_set("batch_id").alias("batch_ids"),
        F.collect_set("missing_value").alias("required_measure_cols")
    )
)

result_dfs = []

for row in execution_groups.toLocalIterator():
    production_line = row["production_line"]
    job_execution_time = row["job_execution_datetetime"]
    batch_ids = row["batch_ids"]
    required_measure_cols = row["required_measure_cols"]

    if not batch_ids or not required_measure_cols:
        continue

    site_table_name = site_table_mapping[production_line]
    adjusted_time_str = (job_execution_time - timedelta(minutes=30)).strftime("%Y-%m-%dT%H:%M:%S")

    monitoring_subset = (
        monitoring_candidates
        .filter(
            (F.col("job_execution_datetetime") == job_execution_time) &
            (F.col("production_line") == production_line)
        )
        .select("production_line", "job_execution_datetetime", "batch_id", "missing_value")
        .dropDuplicates()
    )

    try:
        measurement_pivot_asof_filtered = read_measurement_pivot_asof(
            table_name=site_table_name,
            timestamp_asof=adjusted_time_str,
            batch_ids=batch_ids
        )

        measurement_pivot_long = normalize_measurement_pivot(measurement_pivot_asof_filtered)

        measurement_pivot_presence = (
            measurement_pivot_long
            .filter(
                (F.col("measure_status").isin("OK", "UNFULFILLED")) &
                (F.col("measure_value").isNotNull()) &
                (F.col("measure_name").isin(required_measure_cols))
            )
            .select(
                F.col("prd_line").alias("production_line"),
                "batch_id",
                F.col("measure_name").alias("missing_value")
            )
            .dropDuplicates()
            .alias("p")
        )

        measurement_pivot_check_subset = (
            monitoring_subset.alias("m")
            .join(
                measurement_pivot_presence,
                on=["production_line", "batch_id", "missing_value"],
                how="left"
            )
            .withColumn(
                "is_present_in_measurement_pivot_asof_job_execution",
                F.col("p.batch_id").isNotNull()
            )
            .select(
                "m.*",
                "is_present_in_measurement_pivot_asof_job_execution"
            )
        )

    except AnalysisException as e:
        if "before the earliest version available" in str(e):
            measurement_pivot_check_subset = (
                monitoring_subset
                .withColumn(
                    "is_present_in_measurement_pivot_asof_job_execution",
                    F.lit(False)
                )
            )
        else:
            raise

    result_dfs.append(measurement_pivot_check_subset)

if result_dfs:
    measurement_pivot_check_df = reduce(lambda a, b: a.unionByName(b), result_dfs)
else:
    measurement_pivot_check_df = monitoring_candidates.select(
        "*",
        F.lit(False).alias("is_present_in_measurement_pivot_asof_job_execution")
    )

In [0]:
if len(result_dfs) > 0:
    measurement_pivot_check_candidates = measurement_pivot_check_dfs[0]

    for df in measurement_pivot_check_dfs[1:]:
        measurement_pivot_check_candidates = measurement_pivot_check_candidates.unionByName(df, allowMissingColumns=True)
else:
    measurement_pivot_check_candidates = monitoring_candidates.withColumn(
        "is_present_in_measurement_pivot_asof_job_execution",
        F.lit(False)
    )

In [0]:

measurement_pivot_check =  failure_measure_is_late_tag_drop_with_agg.alias("a").join(
    measurement_pivot_check_candidates.alias("b"),
    (F.col("a.batch_id") == F.col("b.batch_id")) &
    (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")) &
    (F.col("a.missing_value") == F.col("b.missing_value")) &
    (F.col("a.flg_auto") == F.col("b.flg_auto")) &
    (F.col("a.flg_echec") == F.col("b.flg_echec")),
    "left"
).select(
  "a.*",
  F.coalesce(
        F.col("b.is_present_in_measurement_pivot_asof_job_execution"),
        F.lit(False)
    ).alias("is_present_in_measurement_pivot_asof_job_execution")
)

In [0]:
id_alerting = measurement_pivot_check.withColumn(
    "id_alerting",
    F.when(
        (F.col("flg_echec") == 1) &
        (F.col("flg_auto") == 1) &
        (F.col("missing_value") != "NO_VALUE") &
        (F.col("is_present_in_measurement_pivot_asof_job_execution") == True),
        11
    )
    .when((F.col("flg_echec") == 1) & (F.col("flg_auto") == 1) & (F.col("is_missing_occurrences_for_reco") == True), 1)
    .when((F.col("flg_echec") == 1) & (F.col("flg_auto") == 1) & (F.col("is_late_for_reco") == True), 2)
    .when((F.col("flg_echec") == 1) & (F.col("overlap") < 60) & (F.col("missing_value") == "NO_VALUE"), 3)
    .when((F.col("flg_echec") == 1) & (F.col("batch_status") != "error") & (F.col("batch_status") != "pending") & (F.col("missing_value") == "NO_VALUE"), 4)
    .when((F.col("flg_echec") == 1) & (F.col("batch_status") == "error"), 5)
    .when((F.col("flg_echec") == 1) & (F.col("batch_status") == "pending"), 6)
    .when((F.col("flg_echec") == 1) & (F.col("batch_status").isNull()) & (F.col("batch_creation_date") >= F.expr("calculation_interval_after_datetime - INTERVAL 60 MINUTES")), 7)
    .when((F.col("flg_echec") == 1) & (F.col("batch_status").isNull()), 8)
    .when(
        (F.col("missing_value") != "NO_VALUE") &
        (F.col("date_saisie_manuelle").isNotNull()) &
        ((F.col("flg_echec") == 1) | ((F.col("flg_echec") == 0) & (F.col("flg_succes") == 0))) &
        (F.col("id_recommendation").isNull()) &
        (F.col("date_saisie_manuelle") < F.expr("calculation_interval_after_datetime - INTERVAL 30 MINUTES")),
        9
    )
    .when(
        (F.col("missing_value") != "NO_VALUE") &
        (F.col("date_saisie_manuelle").isNotNull()) &
        ((F.col("flg_echec") == 1) | ((F.col("flg_echec") == 0) & (F.col("flg_succes") == 0))) &
        (F.col("id_recommendation").isNull()) &
        (F.col("date_saisie_manuelle") >= F.expr("calculation_interval_after_datetime - INTERVAL 30 MINUTES")) &
        (F.col("date_saisie_manuelle") < F.col("calculation_interval_after_datetime")),
        10
    )
)

In [0]:
# Ajout de la colonne id_batch_tg pour jointure pbi sur table de faits inference monitoring
batch_id_tg_concat = id_alerting.withColumn(
    "id_batch_tg",
    concat_ws("_", "batch_id", "id_parameter_localization")
)


In [0]:
#ajouter la colonne colonne succes echec

#ajouter la colonne colonne succes echec

succes_echec = id_alerting.withColumn(
    "succes_echec",
    F.when((F.col("flg_succes") == 1) & (F.col("flg_echec") == 0), "SUCCES")
     .when((F.col("flg_echec") == 1) & (F.col("flg_succes") == 0), "ECHEC")
    .otherwise(None)
).select(
"id_production_line",
"production_line",
"batch_id",
"id_parameter_localization",
"obsolescence_status",
"batch_status",
"calculation_interval_after_datetime",
"missing_value",
"flg_manual",
"flg_auto",
"type_de_missing_value",
"date_saisie_manuelle",
"succes",
"echec",
"rn",
"flg_succes",
"flg_echec",
"id_alerting",
"succes_echec"
)


Récupérer les dernières informations de chaque recommandation dans la table inference monitoring

In [0]:
flg_manual_auto = max_date.withColumn(
    "has_auto_value",
    F.when(F.size(F.col("automatic_missing_values")) > 0, 1).otherwise(0)
).withColumn(
    "has_manual_value",
    F.when(F.size(F.col("manual_missing_values")) > 0, 1).otherwise(0)
)

raison_manual_auto = flg_manual_auto.withColumn(
    "raison",
    F.when((F.col("has_auto_value") == 1) & (F.col("has_manual_value") == 0), "valeur(s) automatique(s) manquante(s)")
    .when((F.col("has_manual_value") == 1) & (F.col("has_auto_value") == 0), "valeur(s) manuelle(s) manquante(s)")
    .when((F.col("has_manual_value") == 1) & (F.col("has_auto_value") == 1), "valeur(s) manuelle(s) et automatique(s) manquante(s)")
    .otherwise(None))

dim_inference_monitoring = raison_manual_auto.select(
    "batch_id",
    "calculation_interval_after",
    "job_execution_datetetime",
    "target_localization",
    "is_success",
    "batch_status",
    "has_auto_value",
    "has_manual_value",
    "raison"
)

dim_inference_monitoring_id_tg = dim_inference_monitoring.alias("a").join(
    activity_mapping.alias("b"),
    F.col("a.target_localization") == F.col("b.code"),
    "left"
).select("a.*",
         "b.id_parameter_localization")

Maintenant il faut aller récupérer l'id_recommandation depuis la table des reco côté pg

In [0]:
recup_id_reco_pg = succes_reco_count_distinct.alias("a").join(
    dim_inference_monitoring_id_tg.alias("b"),
    (F.col("a.batch_id") == F.col("b.batch_id")) &
    (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")),
    "left"
).select(
    F.col("a.batch_id"),
    ("a.id_recommendation"),

    F.when(F.col("b.calculation_interval_after").isNotNull(), F.col("b.calculation_interval_after"))
     .otherwise(F.col("a.calculation_interval_after_batch_planning"))
     .alias("calculation_interval_after_datetime"), #si valeur absente inference monitoring alors prendre celle issue de pg batch planning

    
    ("b.target_localization"), #col table b
    ("b.is_success"),
    ("b.has_auto_value"),
    ("b.has_manual_value"),
    ("b.raison"),

    ("a.id_parameter_localization"), #col table a
    ("a.date_creation_reco"), 
    ("a.flg_lasted_reco")
)


In [0]:
recup_tg_pg = recup_id_reco_pg.alias("a").join(
    activity_mapping.alias("b"),
    F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization"),
    "left").select("a.batch_id",
                   "a.id_recommendation",
                   "a.calculation_interval_after_datetime",
                   "a.id_parameter_localization",
                   "b.code",
                   "is_success",
                   "has_auto_value",
                   "has_manual_value",
                   "raison",
                   "date_creation_reco",
                   "flg_lasted_reco",
                   F.col("b.code").alias("target_localization_pg"))


Ajout une colonne avec la liste des recommandations concaténées pour garder de la visibilité sur toutes les recommandations générées pour chaque localisation. Comme on a plusieurs reco envoyées parfois, notamment en steep_cycle_1



In [0]:
concat_recos = recup_tg_pg.filter(F.col("flg_lasted_reco") == 0) \
    .groupBy("batch_id", "target_localization_pg") \
    .agg(
        F.concat_ws(", ", F.collect_list("id_recommendation")).alias("concat_id_reco")
    )

latest_recos = recup_tg_pg.filter(F.col("flg_lasted_reco") == 1)

old_reco_concat = latest_recos.join(
    concat_recos,
    on=["batch_id", "target_localization_pg"],
    how="left"
)


In [0]:
#ajout de la colonne type d'echec 

type_echec = old_reco_concat.withColumn("type_echec", F.when(F.col("raison").contains("valeur(s) automatique(s) manquante"), "auto") \
                                            .when(F.col("raison").contains("valeur(s) manuelle(s) manquante(s)"), "manual") \
                                            .when(F.col("raison").contains("valeur(s) manuelle(s) et automatique(s) manquante(s)"), "auto & manual") \
                                            .when((F.col("id_recommendation").isNull()) & ((F.col("has_auto_value") == 0) | (F.col("has_auto_value").isNull())) & ((F.col("has_manual_value") == 0) | (F.col("has_manual_value").isNull())) & (F.col("calculation_interval_after_datetime") <= F.current_timestamp()), "erreur process").otherwise(None))

In [0]:
fact_monitoring = succes_echec.alias("a").join(
    type_echec.alias("b"),
    (F.col("a.batch_id") == F.col("b.batch_id")) &
    (F.col("a.id_parameter_localization") == F.col("b.id_parameter_localization")),
    "left"
).select(
  "a.*",
  "b.code",
  "b.id_recommendation",
  "b.is_success",
  "b.has_auto_value",
  "b.has_manual_value",
  "b.raison",
  "b.date_creation_reco",
  "b.flg_lasted_reco",
  "b.concat_id_reco",
  "b.type_echec"
)


In [0]:
null_values_key_business = fact_monitoring.withColumn(
    "missing_value",
    when(col("missing_value").isNull(), lit("NO_VALUE")).otherwise(col("missing_value"))
)

In [0]:
prd_cell_workshop_batch = null_values_key_business.alias("a").join(
    prd_cell_workshop.alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left").select(
        "a.*",
        "b.steep_vessel1",
        "b.steep_vessel2",
        "b.germ_vessel1",
        "b.germ_vessel2",
        "b.germ_vessel3",
        "b.germ_vessel4",
        "b.germ_vessel5",
        "b.kiln_vessel1",
        "b.kiln_vessel2"
    )

In [0]:

df_with_vessel = (
    prd_cell_workshop_batch
    .withColumn(
        "vessel",
        F.when(
            F.col("code").like("%steeping%"),
            F.concat_ws("_", F.array_distinct(F.array("steep_vessel1", "steep_vessel2")))
        )
        .when(
            F.col("code").like("%germination%"),
            F.concat_ws("_", F.array_distinct(F.array(
                "germ_vessel1", "germ_vessel2", "germ_vessel3", "germ_vessel4", "germ_vessel5"
            )))
        )
        .when(
            F.col("code").like("%kilning%"),
            F.concat_ws("_", F.array_distinct(F.array("kiln_vessel1", "kiln_vessel2")))
        )
    )
    # suppression des anciennes colonnes vessel
    .drop(
        "steep_vessel1", "steep_vessel2",
        "germ_vessel1", "germ_vessel2", "germ_vessel3", "germ_vessel4", "germ_vessel5",
        "kiln_vessel1", "kiln_vessel2"
    )
)

In [0]:
df_date = df_with_vessel.withColumn(
    "calculation_date",
    to_date(col("calculation_interval_after_datetime"))
)

In [0]:
fact_monitoring_auto_manual_entries = df_date.select(
    "id_production_line",
    "batch_id",
    "id_recommendation",
    "calculation_interval_after_datetime",
    "id_parameter_localization",
    "missing_value",
    "flg_manual",
    "flg_auto",
    "type_de_missing_value",
    "date_saisie_manuelle",
    "flg_succes",
    "flg_echec",
    "id_alerting",
    "succes_echec",
    "date_creation_reco",
    "flg_lasted_reco",
    "raison",
    "concat_id_reco",
    "type_echec",
    "has_auto_value",
    "has_manual_value",
    "calculation_date"
)

Ajout de la prd_cell pour ascendance key

Avant l'ingestion des données dans la table cible, on devra valider que les valeurs automatiques et manuelles manquantes mentionnée pour chaque target_localisation de chaque batch, n'ont pas évolué depuis le dernier chargement. Si c'est le cas, il faudra récupérer les informations les plus récentes. Pour cela on doit faire une fonction fenêtre en se basant sur les données déjà présentes dans la table cible. Cette manipulation est importante comme nous utilisons la missing_value comme clé business.

# On prend toutes les paires présentes dans le DataFrame
pairs_to_clean = fact_monitoring_auto_manual_entries \
    .select("batch_id", "id_parameter_localization") \
    .distinct()


In [0]:
# 1. Prépare les paires à remplacer
pairs_to_clean = fact_monitoring_auto_manual_entries \
    .select("batch_id", "id_parameter_localization") \
    .distinct()
pairs_to_clean.createOrReplaceTempView("pairs_to_clean")

In [0]:
current_process= "fact_monitoring_auto_manual_entries"

In [0]:
# 2. DELETE dans la cible (avant le merge/upsert)
spark.sql(f"""
  DELETE FROM {current_catalog}.{current_schema}.{current_process} AS t
  WHERE EXISTS (
    SELECT 1
    FROM pairs_to_clean p
    WHERE t.batch_id = p.batch_id
      AND t.id_parameter_localization = p.id_parameter_localization
  )
""")


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, f"{current_catalog}.{current_schema}.{current_process}")

# Suppression des anciennes lignes correspondantes à ces paires
df_target = delta_table.toDF()
df_cleaned = df_target.join(pairs_to_clean,
                            on=["batch_id", "id_parameter_localization"],
                            how="left_anti")


In [0]:
fact_monitoring_auto_manual_entries.createOrReplaceTempView(f"v_ingestion_{current_process}")

In [0]:
target_fact_monitoring_auto_manual_entries = current_catalog +"."+current_schema+"."+current_process
print(target_fact_monitoring_auto_manual_entries)

In [0]:
all_columns =  fact_monitoring_auto_manual_entries.columns
#display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'batch_id'
    ,'id_parameter_localization'
    ,'missing_value']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    fact_monitoring_auto_manual_entries, 
    target_fact_monitoring_auto_manual_entries, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode or "full" for delete/insert mode
    )